# Jev as a routing-failure judge

**Selected category:** Failure to interpret indirect or underspecified requests.

**Question:** Did the router fail to interpret the indirect or underspecified scheduling request?

`TRUE` means this category is established. `FALSE` can mean a correct route or a different
kind of error. Jev uses exactly the policy and six examples in [the original LLM-only notebook](llm_as_judge.ipynb).
Start with its **21 calibration cases (9 TRUE, 12 FALSE)**; the six examples are excluded.
The 150 validation cases stay reserved until you explicitly enable validation below.

This notebook needs only a TypeSafe key. For the independent LLM-only baseline and
70% confidence fallback, use [the comparison notebook](jev_vs_llm_judge.ipynb).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Colab needs a checkout; locally, open this notebook inside the repository.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    checkout = Path("/content/routing-evals-lab")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "codex/jev-routing-judge",
                        "https://github.com/The-Gen-Academy/Mastering-Agentic-AI-Week4-Session1.git",
                        str(checkout)], check=True)
    os.chdir(checkout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "python-dotenv"], check=True)

folders = [Path.cwd(), Path("routing_agent_evals"), Path("AI-Evals-Principles/routing_agent_evals")]
LAB = next(folder.resolve() for folder in folders if (folder / "judge_lab.py").exists())
os.chdir(LAB)
sys.path.insert(0, str(LAB))

In [ ]:
import os
from getpass import getpass
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from judge_lab import JudgeClient, load_dataset, request_for, batch, comparison_metrics, always_false, save_json, digest, resolved
from notebook_utils import records_for_json
from prompts.llm_as_judge_prompt import (
    FAILURE_CATEGORY, FAILURE_QUESTION, CHOICE_CRITERIA, JEV_JUDGE_PROMPT,
)

# Enable only after freezing the prompt; further tuning requires a new held-out set.
RUN_VALIDATION = False
SPLIT = "validation" if RUN_VALIDATION else "calibration"
load_dotenv(LAB / ".env", override=False)
MODE = os.environ.get("JEV_MODE", "live")  # Replay reads the saved cache without API calls.
assert MODE in {"live", "replay"}
RUN = Path(os.environ.get("JEV_RUN_DIR", LAB / ".runs/indirect-jev" / SPLIT))
key = os.environ.get("TYPESAFE_API_KEY")
if MODE == "live" and not key and IN_COLAB:
    from google.colab import userdata
    try:
        key = userdata.get("TYPESAFE_API_KEY")
    except Exception:
        pass
if MODE == "live" and not key:
    key = getpass("TypeSafe API key (hidden): ")
client = JudgeClient(RUN / "cache", live=(MODE == "live"), api_keys={"jev": key})
client.prompts["jev"] = JEV_JUDGE_PROMPT


## Inspect the question and judge input

Only the request and predicted intent reach Jev. The shared rubric requires indirect
or incomplete expression, a prediction ruled out under every defensible policy reading,
and a mismatch in recovering the scheduling meaning. Polarity errors and unsupported
patient-specific operations are separate categories. Human labels never enter requests.

Jev uses binary **Choice** to return TRUE/FALSE, probabilities and native confidence.
The helper validates that response; invalid answers remain unresolved. A confidence
of 0.70 corresponds to a winning probability of 0.85, not 70% measured accuracy.


In [ ]:
rows = load_dataset(LAB / "data", split=SPLIT)
print("Category:", FAILURE_CATEGORY)
print("Question:", FAILURE_QUESTION)
display(pd.DataFrame(CHOICE_CRITERIA.items(), columns=["answer", "meaning"]))
print("Split:", SPLIT, "| cases:", len(rows))
print("State sent to Jev:", request_for(rows[0], "jev")["state"])
display(pd.DataFrame(rows)[["id", "user_query", "predicted_intent"]].head(3))


In [ ]:
jev = batch(rows, lambda row: client.call(row, "jev"))
display(pd.DataFrame([jev[rows[0]["id"]]]).reindex(columns=["label", "confidence", "probabilities", "error"]))

## Compare with the references and log results

Check TRUE recall and false alarms alongside agreement. Score both methods on the same
resolved cases; report coverage and unresolved counts separately. Costs include failed
attempts and may be estimates; unknown costs are displayed as unavailable.


In [ ]:
comparison = pd.DataFrame(comparison_metrics(rows, {"Always FALSE": always_false(rows), "Jev": jev}))
table = comparison[["method", "scored_n", "coverage", "accuracy", "precision_TRUE", "recall_TRUE", "f1_TRUE", "FP", "FN", "unresolved", "cost_usd"]].copy()
table["cost_usd"] = table.cost_usd.where(comparison.unknown_cost_attempts.eq(0))
display(table.round(4))
RUN.mkdir(parents=True, exist_ok=True)
summary = {
    "split": SPLIT, "category": FAILURE_CATEGORY, "question": FAILURE_QUESTION,
    "dataset_hash": digest(rows), "evaluation_ids": [r["id"] for r in rows],
    "prompt_hash": digest([JEV_JUDGE_PROMPT, CHOICE_CRITERIA]),
    "model": client.models["jev"], "comparison": records_for_json(comparison),
    "scoring_ids": [r["id"] for r in rows if resolved(jev[r["id"]])],
}
if MODE == "replay":
    import json
    if json.loads((RUN / "manifest.json").read_text()) != summary:
        raise ValueError("Replay differs from the saved dataset, prompt, model or results.")
else:
    save_json(RUN / "manifest.json", summary)
comparison.to_csv(RUN / "comparison.csv", index=False)
pd.DataFrame([dict(result, category=FAILURE_CATEGORY, question=FAILURE_QUESTION, split=SPLIT)
              for result in jev.values()]).to_csv(RUN / "results.csv", index=False)
print("Outputs:", RUN)
